In [ ]:
import numpy as np

from implementations import (
    mean_squared_error_gd,
    mean_squared_error_sgd,
    least_squares,
    ridge_regression,
    logistic_regression,
    reg_logistic_regression,
)

from helpers import load_csv_data, create_csv_submission

DATA_PATH = "data/dataset/"

SEED = 67
VALIDATION_FRACTION = 0.20

# settings
MSE_GD_MAX_ITERS = 500
MSE_GD_GAMMA = 0.01

MSE_SGD_MAX_ITERS = 100_000
MSE_SGD_GAMMA = 1e-5

RIDGE_LAMBDA = 1e-7

LOGISTIC_MAX_ITERS = 1000
LOGISTIC_GAMMA = 0.03
LOGISTIC_LAMBDA = 0.01

In [2]:
x_train_raw, x_test_raw, y_train, train_ids, test_ids = load_csv_data(
    DATA_PATH,
    sub_sample=False
)

y_train = np.asarray(y_train).ravel()

print("Train:", x_train_raw.shape)
print("Test:", x_test_raw.shape)
print("Labels:", np.unique(y_train, return_counts=True))

# Stratified split
rng = np.random.default_rng(SEED)

positive_idx = np.where(y_train == 1)[0]
negative_idx = np.where(y_train == -1)[0]

rng.shuffle(positive_idx)
rng.shuffle(negative_idx)

n_pos_val = int(len(positive_idx) * VALIDATION_FRACTION)
n_neg_val = int(len(negative_idx) * VALIDATION_FRACTION)

val_idx = np.concatenate([
    positive_idx[:n_pos_val],
    negative_idx[:n_neg_val]
])

train_idx = np.concatenate([
    positive_idx[n_pos_val:],
    negative_idx[n_neg_val:]
])

rng.shuffle(train_idx)
rng.shuffle(val_idx)

x_fit_raw = x_train_raw[train_idx]
y_fit = y_train[train_idx]

x_val_raw = x_train_raw[val_idx]
y_val = y_train[val_idx]

print("Fit split:", x_fit_raw.shape)
print("Validation split:", x_val_raw.shape)

Train: (328135, 321)
Test: (109379, 321)
Labels: (array([-1,  1]), array([299160,  28975]))
Fit split: (262508, 321)
Validation split: (65627, 321)


In [3]:
def fit_preprocessor(X):

    # Remove features that are entirely missing
    keep = np.any(np.isfinite(X), axis=0)
    X = X[:, keep].astype(float, copy=True)

    # Treat non-finite values as missing
    X[~np.isfinite(X)] = np.nan

    means = np.nanmean(X, axis=0)
    stds = np.nanstd(X, axis=0)

    # Avoid division by zero or NaN in case of constant features
    stds[~np.isfinite(stds) | (stds == 0)] = 1.0

    return keep, means, stds


def transform_features(X, keep, means, stds):
    X = X[:, keep].astype(float, copy=True)
    X[~np.isfinite(X)] = np.nan

    # Impute missing values using training means
    X = np.where(np.isnan(X), means, X)

    X = (X - means) / stds

    # Add bias term
    X = np.column_stack([np.ones(X.shape[0]), X])

    return X


# Validation preprocessing: fit on the fitting split only
keep_val, means_val, stds_val = fit_preprocessor(x_fit_raw)

X_fit = transform_features(
    x_fit_raw, keep_val, means_val, stds_val
)

X_val = transform_features(
    x_val_raw, keep_val, means_val, stds_val
)

print("Processed fit:", X_fit.shape)
print("Processed validation:", X_val.shape)

Processed fit: (262508, 322)
Processed validation: (65627, 322)


In [4]:
def fit_mse_gd(y, X):
    return mean_squared_error_gd(
        y, X, np.zeros(X.shape[1]),
        max_iters=MSE_GD_MAX_ITERS,
        gamma=MSE_GD_GAMMA
    )


def fit_mse_sgd(y, X):
    return mean_squared_error_sgd(
        y, X, np.zeros(X.shape[1]),
        max_iters=MSE_SGD_MAX_ITERS,
        gamma=MSE_SGD_GAMMA
    )


def fit_least_squares(y, X):
    return least_squares(y, X)


def fit_ridge(y, X):
    return ridge_regression(y, X, RIDGE_LAMBDA)


def fit_logistic(y, X):
    return logistic_regression(
        y, X, np.zeros(X.shape[1]),
        max_iters=LOGISTIC_MAX_ITERS,
        gamma=LOGISTIC_GAMMA
    )


def fit_reg_logistic(y, X):
    return reg_logistic_regression(
        y, X, LOGISTIC_LAMBDA, np.zeros(X.shape[1]),
        max_iters=LOGISTIC_MAX_ITERS,
        gamma=LOGISTIC_GAMMA
    )


models = {
    "mse_gd": fit_mse_gd,
    "mse_sgd": fit_mse_sgd,
    "least_squares": fit_least_squares,
    "ridge": fit_ridge,
    "logistic": fit_logistic,
    "reg_logistic": fit_reg_logistic,
}

In [5]:
def find_best_accuracy_threshold(y_true, scores):

    candidates = np.unique(
        np.quantile(scores, np.linspace(0, 1, 1001))
    )

    best_threshold = 0.0
    best_accuracy = -1.0

    for threshold in candidates:
        predictions = np.where(scores >= threshold, 1, -1)
        accuracy = np.mean(predictions == y_true)

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_threshold = threshold

    return best_threshold, best_accuracy


def predict_labels(scores, threshold):
    return np.where(scores >= threshold, 1, -1)

In [8]:
results = {}

# Fit preprocessing on ALL labelled training data for final models
full_keep, full_means, full_stds = fit_preprocessor(x_train_raw)

X_full = transform_features(
    x_train_raw, full_keep, full_means, full_stds
)

X_test = transform_features(
    x_test_raw, full_keep, full_means, full_stds
)

for name, fit_model in models.items():
    print("MODEL:", name)

    # 1. Fit on the fitting split
    w_val, val_train_loss = fit_model(y_fit, X_fit)

    # 2. Get validation scores and choose accuracy threshold
    val_scores = X_val @ w_val

    if not np.isfinite(val_scores).all():
        print("Validation scores contain NaN or infinity.")

    threshold, val_accuracy = find_best_accuracy_threshold(
        y_val, val_scores
    )

    print(f"Validation training loss: {val_train_loss:.8f}")
    print(f"Chosen threshold:         {threshold:.8f}")
    print(f"Validation accuracy:      {val_accuracy:.5f}")

    # 3. Retrain on the entire labelled dataset
    w_final, full_train_loss = fit_model(y_train, X_full)

    # 4. Predict test labels
    predictions = X_test @ w_final

    if not np.isfinite(predictions).all():
        print("Predictions contains NaN or infinity.")

    test_predictions = predict_labels(predictions, threshold)

    # 5. Save one submission per model
    output_path = f"submission_{name}.csv"

    create_csv_submission(
        test_ids,
        test_predictions,
        str(output_path)
    )

    results[name] = {
        "threshold": threshold,
        "validation_accuracy": val_accuracy,
        "validation_loss": val_train_loss,
        "full_training_loss": full_train_loss,
        "positive_predictions": int(np.sum(test_predictions == 1)),
        "negative_predictions": int(np.sum(test_predictions == -1)),
        "file": str(output_path),
    }

    print("Full training loss:", full_train_loss)
    print("Test predictions:", len(test_predictions))
    print("Predicted classes:", np.unique(
        test_predictions, return_counts=True
    ))
    print("Saved:", output_path)


print("\nFinished generating submissions.")

MODEL: mse_gd
Validation training loss: 0.13788921
Chosen threshold:         -0.29973476
Validation accuracy:      0.91366
Full training loss: 0.13772940989949103
Test predictions: 109379
Predicted classes: (array([-1,  1]), array([107650,   1729]))
Saved: submission_mse_gd.csv
MODEL: mse_sgd
Validation training loss: 0.18520534
Chosen threshold:         -0.06293152
Validation accuracy:      0.91299
Full training loss: 0.1845320636668948
Test predictions: 109379
Predicted classes: (array([-1,  1]), array([107434,   1945]))
Saved: submission_mse_sgd.csv
MODEL: least_squares
Validation training loss: 0.13668351
Chosen threshold:         -0.17907897
Validation accuracy:      0.91340
Full training loss: 0.13650496446831767
Test predictions: 109379
Predicted classes: (array([-1,  1]), array([108626,    753]))
Saved: submission_least_squares.csv
MODEL: ridge
Validation training loss: 0.13668355
Chosen threshold:         -0.17926130
Validation accuracy:      0.91340
Full training loss: 0.1365

In [9]:

def get_validation_accuracy(item):
    name, result = item
    return result["validation_accuracy"]


print(
    f"{'Model':<18} {'Val acc':>9} {'Threshold':>12} "
    f"{'Positives':>11}"
)

sorted_results = sorted(
    results.items(),
    key=get_validation_accuracy,
    reverse=True
)

for name, result in sorted_results:
    print(
        f"{name:<18} "
        f"{result['validation_accuracy']:>9.4f} "
        f"{result['threshold']:>12.5f} "
        f"{result['positive_predictions']:>11} "
    )


Model                Val acc    Threshold   Positives
logistic              0.9140     -0.04114        1311 
reg_logistic          0.9138     -0.13571        1435 
mse_gd                0.9137     -0.29973        1729 
least_squares         0.9134     -0.17908         753 
ridge                 0.9134     -0.17926         756 
mse_sgd               0.9130     -0.06293        1945 
